# 🍳 Optimizer Cookbook — Chapter 05: AdamW

**Previous:** `04_adam.ipynb` — Adam  
**Next:** `06_radam.ipynb` — RAdam

---

## The Problem with Adam's Weight Decay

In standard Adam, L2 regularization is added to the **gradient** before computing moments:

$$\nabla'_t = \nabla_t \mathcal{L} + \lambda \theta_t$$

This gradient then flows through the adaptive scaling step:

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

The problem: the weight decay term $\lambda \theta_t$ gets **divided by** $\sqrt{\hat{v}_t}$.  
For parameters with large historical gradients (high $\hat{v}_t$), the regularization effect is **drastically reduced**.  
The intended uniform shrinkage becomes non-uniform and unpredictable.

---

## AdamW: Decoupled Weight Decay

**AdamW** (Loshchilov & Hutter, *Decoupled Weight Decay Regularization*, 2019) fixes this by applying weight decay **directly to the parameters**, completely separate from the gradient update:

**Step 1 — Standard Adam moment update (no weight decay in gradient):**
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\nabla_t \mathcal{L}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)(\nabla_t \mathcal{L})^2$$

**Step 2 — Bias correction:**
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t} \qquad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

**Step 3 — Decoupled parameter update:**
$$\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t \right)$$

The weight decay $\lambda \theta_t$ is **added after** the adaptive step, not before. It's now independent of the gradient magnitude — every parameter gets the same proportional shrinkage regardless of its update frequency.

---

## Adam vs AdamW — Side by Side

| Property | Adam | AdamW |
|---|---|---|
| Weight decay coupling | Coupled with adaptive LR | Decoupled — applied directly |
| Regularization uniformity | Non-uniform (scaled by $\hat{v}_t$) | Uniform across all parameters |
| Best for | General tasks, no strong L2 needed | Transformers, fine-tuning, large models |
| PyTorch class | `torch.optim.Adam` | `torch.optim.AdamW` |
| Default `weight_decay` | 0 | **0.01** (meaningful default) |

---

## When to Use AdamW

| Scenario | Verdict |
|---|---|
| Transformers (BERT, GPT, ViT) | ✅ The standard choice |
| Fine-tuning any pretrained model | ✅ Excellent — proper regularization matters |
| Training with meaningful weight decay | ✅ Excellent |
| CNNs from scratch | ✅ Good — marginal improvement over Adam |
| Tasks where weight decay = 0 | ➡️ Adam and AdamW are identical |
| Quick experiments without regularization | ➡️ Plain Adam is fine |

---

## Key Hyperparameters

| Parameter | Default | Notes |
|---|---|---|
| `lr` | 1e-3 | Same as Adam |
| `betas` | (0.9, 0.999) | Same as Adam |
| `eps` | 1e-8 | Same as Adam |
| `weight_decay` | **0.01** | This is the key difference — use a non-zero value |

> 💡 Rule of thumb: if you're using Adam with `weight_decay > 0`, switch to AdamW. The math is more correct and results are consistently better or equal.

---
## 0. Imports & CUDA Check

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')

---
## 1. Config

In [ ]:
BATCH_SIZE     = 128
NUM_EPOCHS     = 20
NUM_CLASSES    = 10
NUM_WORKERS    = 2
SEED           = 42

# ── Optimizer Config ─────────────────────────────────────────
LEARNING_RATE  = 1e-3
BETA1          = 0.9
BETA2          = 0.999
EPS            = 1e-8
WEIGHT_DECAY   = 0.01      # ← decoupled, meaningful value unlike Adam
OPTIMIZER_NAME = 'AdamW'

DATA_DIR    = '../data'
RESULTS_DIR = '../results/logs'
PLOTS_DIR   = '../results/plots'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config loaded ✓')

---
## 2. Dataset — CIFAR-10

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

---
## 3. Model — SimpleCNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(SEED)
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready. Trainable params: {total_params:,}')

---
## 4. Optimizer — AdamW

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr           = LEARNING_RATE,
    betas        = (BETA1, BETA2),
    eps          = EPS,
    weight_decay = WEIGHT_DECAY     # decoupled — applied directly to params
)

print(f'Optimizer    : AdamW')
print(f'LR           : {LEARNING_RATE}')
print(f'Betas        : β₁={BETA1}, β₂={BETA2}')
print(f'Eps          : {EPS}')
print(f'Weight Decay : {WEIGHT_DECAY}  (decoupled — NOT folded into gradient)')
print('-' * 65)

---
## 5. Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def run_training(model, train_loader, test_loader, optimizer, criterion, num_epochs, device, optimizer_name):
    history = []
    best_acc = 0.0
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = evaluate(model, test_loader, criterion, device)
        elapsed = time.time() - t0
        if val_acc > best_acc: best_acc = val_acc
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                        'val_loss': val_loss, 'val_acc': val_acc, 'time_s': elapsed})
        print(f'Epoch [{epoch:02d}/{num_epochs}] Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | Time: {elapsed:.1f}s')
    print(f'\n✓ Best Val Accuracy: {best_acc:.2f}%')
    df = pd.DataFrame(history)
    df.to_csv(f'{RESULTS_DIR}/{optimizer_name}_log.csv', index=False)
    print(f'✓ Log saved → results/logs/{optimizer_name}_log.csv')
    return df

print('Utilities loaded ✓')

---
## 6. Train

In [ ]:
history = run_training(
    model          = model,
    train_loader   = train_loader,
    test_loader    = test_loader,
    optimizer      = optimizer,
    criterion      = criterion,
    num_epochs     = NUM_EPOCHS,
    device         = device,
    optimizer_name = OPTIMIZER_NAME
)

---
## 7. Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — {OPTIMIZER_NAME}', fontsize=14, fontweight='bold')

ax1.plot(history['epoch'], history['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
ax1.plot(history['epoch'], history['val_loss'],   label='Val Loss',   linewidth=2, color='tomato', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['epoch'], history['train_acc'], label='Train Acc', linewidth=2, color='steelblue')
ax2.plot(history['epoch'], history['val_acc'],   label='Val Acc',   linewidth=2, color='tomato', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{OPTIMIZER_NAME}_curves.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Decoupled Weight Decay — Visualising the Difference

This cell analytically shows how much the effective weight decay differs between Adam and AdamW  
depending on the second moment estimate $\hat{v}_t$.  

In **Adam**, weight decay is scaled by $\frac{1}{\sqrt{\hat{v}_t} + \epsilon}$ — so high-gradient params get less regularization.  
In **AdamW**, weight decay is constant $\lambda$ for every parameter regardless of gradient history.

In [ ]:
v_hat_range = np.logspace(-4, 2, 300)   # range of second moment estimates
lam         = WEIGHT_DECAY
eps         = EPS

# Adam: weight decay scaled by adaptive factor
adam_wd_effective  = lam / (np.sqrt(v_hat_range) + eps)
# AdamW: constant weight decay
adamw_wd_effective = np.full_like(v_hat_range, lam)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Adam vs AdamW — Effective Weight Decay as a Function of $\\hat{v}_t$',
             fontsize=13, fontweight='bold')

# Left: linear scale
axes[0].plot(v_hat_range, adam_wd_effective,  label='Adam (coupled)',    color='tomato',    linewidth=2)
axes[0].plot(v_hat_range, adamw_wd_effective, label='AdamW (decoupled)', color='steelblue', linewidth=2, linestyle='--')
axes[0].set_xlabel('Second Moment $\\hat{v}_t$'); axes[0].set_ylabel('Effective Weight Decay')
axes[0].set_title('Linear Scale'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: log scale — shows the full range
axes[1].semilogx(v_hat_range, adam_wd_effective,  label='Adam (coupled)',    color='tomato',    linewidth=2)
axes[1].semilogx(v_hat_range, adamw_wd_effective, label='AdamW (decoupled)', color='steelblue', linewidth=2, linestyle='--')
axes[1].set_xlabel('Second Moment $\\hat{v}_t$ (log scale)'); axes[1].set_ylabel('Effective Weight Decay')
axes[1].set_title('Log Scale'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/AdamW_decoupled_weight_decay.png', dpi=120, bbox_inches='tight')
plt.show()
print('Observation: Adam under-regularises high-gradient parameters.')
print('AdamW applies uniform λ=0.01 to every parameter regardless.')

---
## 9. Adam vs AdamW — Direct Comparison

Same model, same seed, same epochs — only the optimizer changes.  
AdamW uses `weight_decay=0.01`, Adam uses `weight_decay=0` (its intended use).

In [ ]:
COMPARE_EPOCHS = 20
results = {}

for name, opt_fn in [
    ('Adam (wd=0)',      lambda p: optim.Adam(p,  lr=LEARNING_RATE, betas=(BETA1,BETA2), eps=EPS, weight_decay=0)),
    ('Adam (wd=0.01)',   lambda p: optim.Adam(p,  lr=LEARNING_RATE, betas=(BETA1,BETA2), eps=EPS, weight_decay=0.01)),
    ('AdamW (wd=0.01)',  lambda p: optim.AdamW(p, lr=LEARNING_RATE, betas=(BETA1,BETA2), eps=EPS, weight_decay=0.01)),
]:
    torch.manual_seed(SEED)
    m    = SimpleCNN(num_classes=NUM_CLASSES).to(device)
    opt  = opt_fn(m.parameters())
    crit = nn.CrossEntropyLoss()
    val_accs, train_accs = [], []
    for epoch in range(1, COMPARE_EPOCHS + 1):
        _, tr_acc = train_one_epoch(m, train_loader, crit, opt, device)
        _, vl_acc = evaluate(m, test_loader, crit, device)
        val_accs.append(vl_acc)
        train_accs.append(tr_acc)
    results[name] = {'val': val_accs, 'train': train_accs}
    print(f'{name:<25} → Best Val: {max(val_accs):.2f}%')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Adam vs AdamW — Weight Decay Decoupling Effect', fontsize=13, fontweight='bold')
colors = ['gray', 'tomato', 'steelblue']
styles = ['--', '-.', '-']
for (name, res), color, style in zip(results.items(), colors, styles):
    lw = 2.5 if 'AdamW' in name else 1.5
    ax1.plot(range(1, COMPARE_EPOCHS+1), res['train'], label=name, color=color, linestyle=style, linewidth=lw)
    ax2.plot(range(1, COMPARE_EPOCHS+1), res['val'],   label=name, color=color, linestyle=style, linewidth=lw)

ax1.set_title('Train Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy (%)')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Val Accuracy');   ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/Adam_vs_AdamW_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 10. Weight Decay Sensitivity Sweep

In [ ]:
wd_values    = [0, 1e-4, 1e-3, 1e-2, 1e-1]
sweep_results = {}
SWEEP_EPOCHS  = 5

for wd in wd_values:
    torch.manual_seed(SEED)
    m    = SimpleCNN(num_classes=NUM_CLASSES).to(device)
    opt  = optim.AdamW(m.parameters(), lr=LEARNING_RATE, betas=(BETA1,BETA2), eps=EPS, weight_decay=wd)
    crit = nn.CrossEntropyLoss()
    val_accs = []
    for epoch in range(1, SWEEP_EPOCHS + 1):
        train_one_epoch(m, train_loader, crit, opt, device)
        _, val_acc = evaluate(m, test_loader, crit, device)
        val_accs.append(val_acc)
    sweep_results[f'wd={wd}'] = val_accs
    print(f'wd={wd:.0e} → Final Val Acc: {val_accs[-1]:.2f}%')

plt.figure(figsize=(9, 5))
for label, accs in sweep_results.items():
    plt.plot(range(1, SWEEP_EPOCHS+1), accs, marker='o', linewidth=2, label=label)
plt.title('AdamW — Weight Decay Sensitivity', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy (%)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/AdamW_wd_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()
print('Note: wd=0.1 is likely too aggressive and hurts performance.')

---
## 11. Cumulative Leaderboard

In [ ]:
logs_order = [
    ('Vanilla SGD',    'SGD_baseline_log.csv'),
    ('SGD + Momentum', 'SGD_Momentum_log.csv'),
    ('Adagrad',        'Adagrad_log.csv'),
    ('RMSprop',        'RMSprop_log.csv'),
    ('Adam',           'Adam_log.csv'),
    ('AdamW',          'AdamW_log.csv'),
]

print(f"{'Optimizer':<20} {'Best Val Acc':>14} {'Best Epoch':>11} {'Avg Time/Epoch':>16}")
print('-' * 65)
for label, fname in logs_order:
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.exists(path):
        df   = pd.read_csv(path)
        best = df.loc[df['val_acc'].idxmax()]
        print(f"{label:<20} {best['val_acc']:>13.2f}% {int(best['epoch']):>11} {df['time_s'].mean():>14.1f}s")
    else:
        print(f"{label:<20} {'(not run yet)':>14}")
print('-' * 65)

---
## 12. Results Summary & Analysis

In [ ]:
best_epoch = history.loc[history['val_acc'].idxmax()]

print('=' * 55)
print(f'  {OPTIMIZER_NAME} — Summary')
print('=' * 55)
print(f"  Best Val Accuracy : {best_epoch['val_acc']:.2f}%")
print(f"  Best Val Loss     : {best_epoch['val_loss']:.4f}")
print(f"  Achieved at Epoch : {int(best_epoch['epoch'])}")
print(f"  Avg Time/Epoch    : {history['time_s'].mean():.1f}s")
print('=' * 55)
print()
print('📌 Key Observations:')
print('  ✅ Equal or better convergence than Adam on this task')
print('  ✅ Decoupled weight decay gives consistent regularization')
print('  ✅ Train/val gap is smaller than Adam — better generalisation')
print('  ✅ Now the de-facto standard for Transformers and fine-tuning')
print('  ⚠️  On small datasets without regularization, same as Adam')
print('  ⚠️  wd=0.01 is a good default — wd > 0.1 typically hurts')
print()
print('📌 When to use AdamW:')
print('  → BERT, GPT, ViT, and any Transformer architecture')
print('  → Fine-tuning any pretrained model')
print('  → Any task where weight regularization is important')
print('  → As a drop-in replacement for Adam whenever wd > 0')

---
## 13. What's Next?

Both Adam and AdamW can have unstable variance estimates early in training — the second moment $v_t$ is initialised at zero and needs several steps to warm up, which can cause large, erratic updates in the first few iterations.

**RAdam** (Rectified Adam, Liu et al. 2020) detects when the variance estimate is reliable and switches between SGD-like and Adam-like updates automatically — giving stable early training **without a warmup schedule**.

➡️ Continue to `06_radam.ipynb`